# Grid PM10 → Shapefile 변환

V5 ST-GNN ambient PM10 예측값을 250m 격자 폴리곤 shapefile로 내보냅니다.

**출력 파일 (ZIP):**
- `grid_pm10_YYYYMMDD_HH00.zip` — .shp, .dbf, .shx, .prj, .geojson 포함

**좌표계:** EPSG:5179 (Korea 2000 Unified CS, 단위: 미터)

---
셀 2의 설정만 바꾸고 전체 실행하세요.

In [ ]:
import os, sys, json, zipfile, struct, warnings
import numpy as np
import pandas as pd
import shapefile          # pyshp
warnings.filterwarnings('ignore')

ROOT     = '/workspace/ST-GNN Modeling'
OUT      = os.path.join(ROOT, 'Visualization/outputs')
GRID_CSV = '/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/격자_250m_4326.csv'
TS_LOOKUP = os.path.join(ROOT, 'RoadExtension_V2/checkpoints/v5_ts_lookup.csv')
os.makedirs(OUT, exist_ok=True)

# V5 데이터 로드
v5_data = {
    'train': np.load(os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_train.npy')),
    'val':   np.load(os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_val.npy')),
    'test':  np.load(os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/V5-base/grid_pm10_test.npy')),
}
ts_lookup = pd.read_csv(TS_LOOKUP)
ts_lookup['dt'] = pd.to_datetime(ts_lookup['timestamp'])

# 격자 데이터
grid_df = pd.read_csv(GRID_CSV)

print('로드 완료  |  격자 {:,}개'.format(len(grid_df)))
print('test 기간: {} ~ {}'.format(
    ts_lookup[ts_lookup['split']=='test']['dt'].min().strftime('%Y-%m-%d'),
    ts_lookup[ts_lookup['split']=='test']['dt'].max().strftime('%Y-%m-%d')))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ★ 설정 ★
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TARGET_DT = '2025-10-01 10:00:00'

# 내보낼 범위: 'all' = 서울 전체, 'gangnam' = 강남구만
EXPORT_AREA = 'all'     # 'all' 또는 'gangnam'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BBOX = {
    'all':     dict(lat_min=37.4, lat_max=37.72, lon_min=126.7, lon_max=127.25),
    'gangnam': dict(lat_min=37.485, lat_max=37.540, lon_min=127.005, lon_max=127.100),
}
print('설정:', TARGET_DT, '|', EXPORT_AREA)

In [ ]:
# ── V5 PM10 획득 ─────────────────────────────────────────────────────────
dt    = pd.Timestamp(TARGET_DT)
match = ts_lookup[ts_lookup['dt'] == dt]
if len(match) == 0:
    match = ts_lookup.loc[[(ts_lookup['dt'] - dt).abs().idxmin()]]
    print('가장 가까운 시간 사용:', match['dt'].values[0])
row      = match.iloc[0]
pm10_all = v5_data[row['split']][int(row['local_idx'])]   # (G=10125,)
actual_dt = pd.Timestamp(row['dt'])
print('추출 시간:', actual_dt)

# ── 범위 필터 ─────────────────────────────────────────────────────────────
bbox = BBOX[EXPORT_AREA]
mask = ((grid_df['lat'] >= bbox['lat_min']) & (grid_df['lat'] <= bbox['lat_max']) &
        (grid_df['lon'] >= bbox['lon_min']) & (grid_df['lon'] <= bbox['lon_max']))
sel_df  = grid_df[mask].copy().reset_index(drop=True)
sel_pm10 = pm10_all[grid_df[mask].index]

print('내보낼 격자 수: {:,}개'.format(len(sel_df)))
print('PM10 범위: {:.2f} ~ {:.2f}  mean={:.2f} μg/m³'.format(
    sel_pm10.min(), sel_pm10.max(), sel_pm10.mean()))

In [ ]:
# ── Shapefile 생성 ────────────────────────────────────────────────────────
CELL_SIZE = 250    # 250m 격자
HALF      = CELL_SIZE / 2

ts_str   = actual_dt.strftime('%Y%m%d_%H00')
shp_name = 'grid_pm10_{}'.format(ts_str)
shp_dir  = os.path.join(OUT, shp_name)
os.makedirs(shp_dir, exist_ok=True)

# shapefile 쓰기 (폴리곤 타입)
w = shapefile.Writer(os.path.join(shp_dir, shp_name),
                     shapeType=shapefile.POLYGON)

# 속성 필드 정의
w.field('CELL_ID',  'C', size=20)        # 격자 ID
w.field('PM10',     'N', decimal=4)      # PM10 μg/m³
w.field('LAT',      'N', decimal=6)      # 중심 위도
w.field('LON',      'N', decimal=6)      # 중심 경도
w.field('CELL_X',   'N', decimal=0)      # 투영 X (m)
w.field('CELL_Y',   'N', decimal=0)      # 투영 Y (m)

for i, (_, row_g) in enumerate(sel_df.iterrows()):
    cx = float(row_g['CELL_X'])
    cy = float(row_g['CELL_Y'])
    pm = float(sel_pm10[i])

    # 250m 격자 폴리곤 (EPSG:5179 투영 좌표, 시계방향)
    ring = [
        [cx - HALF, cy - HALF],
        [cx - HALF, cy + HALF],
        [cx + HALF, cy + HALF],
        [cx + HALF, cy - HALF],
        [cx - HALF, cy - HALF],   # 닫기
    ]
    w.poly([ring])
    w.record(
        CELL_ID = str(row_g['CELL_ID']),
        PM10    = round(pm, 4),
        LAT     = round(float(row_g['lat']), 6),
        LON     = round(float(row_g['lon']), 6),
        CELL_X  = int(cx),
        CELL_Y  = int(cy),
    )

w.close()
print('SHP 작성 완료: {:,}개 폴리곤'.format(len(sel_df)))

# .prj 파일 — EPSG:5179 Korea 2000 Unified CS
prj_wkt = (
    'PROJCS["Korea 2000 / Unified CS",'
    'GEOGCS["Korea 2000",'
    'DATUM["Geocentric_datum_of_Korea",'
    'SPHEROID["GRS 1980",6378137,298.257222101]],'
    'PRIMEM["Greenwich",0],'
    'UNIT["degree",0.0174532925199433]],'
    'PROJECTION["Transverse_Mercator"],'
    'PARAMETER["latitude_of_origin",38],'
    'PARAMETER["central_meridian",127],'
    'PARAMETER["scale_factor",1],'
    'PARAMETER["false_easting",200000],'
    'PARAMETER["false_northing",600000],'
    'UNIT["metre",1],'
    'AUTHORITY["EPSG","5179"]]'
)
with open(os.path.join(shp_dir, shp_name + '.prj'), 'w') as f:
    f.write(prj_wkt)
print('PRJ 작성 완료: EPSG:5179')

In [ ]:
# ── GeoJSON 추가 생성 (WGS84, 범용 포맷) ─────────────────────────────────

# 투영 좌표 → WGS84 근사 변환 (±0.0001도 오차 수준)
# 정확한 변환 없이 격자 중심 lat/lon ± 약 0.00113도 (250m ≈ 125m/반경)
LAT_DEG_PER_M = 1 / 111320.0

def lon_deg_per_m(lat_deg):
    import math
    return 1 / (111320.0 * math.cos(math.radians(lat_deg)))

features = []
for i, (_, row_g) in enumerate(sel_df.iterrows()):
    lat = float(row_g['lat'])
    lon = float(row_g['lon'])
    pm  = float(sel_pm10[i])
    dlat = HALF * LAT_DEG_PER_M
    dlon = HALF * lon_deg_per_m(lat)

    coords = [[
        [lon - dlon, lat - dlat],
        [lon - dlon, lat + dlat],
        [lon + dlon, lat + dlat],
        [lon + dlon, lat - dlat],
        [lon - dlon, lat - dlat],
    ]]
    features.append({
        'type': 'Feature',
        'geometry': {'type': 'Polygon', 'coordinates': coords},
        'properties': {
            'CELL_ID': str(row_g['CELL_ID']),
            'PM10':    round(pm, 4),
            'lat':     round(lat, 6),
            'lon':     round(lon, 6),
        }
    })

geojson = {
    'type': 'FeatureCollection',
    'crs':  {'type': 'name', 'properties': {'name': 'EPSG:4326'}},
    'metadata': {
        'datetime': str(actual_dt),
        'model':    'HiddenExtension V5-base',
        'variable': 'Ambient PM10 (ug/m3)',
        'grid_size_m': 250,
        'n_cells':  len(features),
    },
    'features': features
}

geojson_path = os.path.join(shp_dir, shp_name + '.geojson')
with open(geojson_path, 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False)
print('GeoJSON 작성 완료: {:,}개 피처'.format(len(features)))

In [ ]:
# ── CSV 추가 (가장 범용) ──────────────────────────────────────────────────
csv_df = sel_df[['CELL_ID','lat','lon','CELL_X','CELL_Y']].copy()
csv_df['PM10_ugm3'] = sel_pm10
csv_df['datetime']  = str(actual_dt)

csv_path = os.path.join(shp_dir, shp_name + '.csv')
csv_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('CSV 작성 완료')

# ── ZIP 패키징 ────────────────────────────────────────────────────────────
zip_path = os.path.join(OUT, shp_name + '.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(shp_dir):
        zf.write(os.path.join(shp_dir, fname), arcname=fname)

zip_kb = os.path.getsize(zip_path) // 1024
print('\n=== 내보내기 완료 ===')
print('ZIP:', zip_path)
print('크기: {:,} KB'.format(zip_kb))
print()
print('포함 파일:')
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        print('  {:35s}  {:,} KB'.format(info.filename, info.file_size // 1024))
print()
print('QGIS/ArcGIS: .shp 파일 열기 (EPSG:5179)')
print('웹 GIS:      .geojson 파일 열기 (WGS84)')
print('엑셀/분석:   .csv 파일 열기')